In [1]:
import os
import sqlite3
import pandas as pd

# =====================================================================
# 🛠️ 准备 Cell：【直击现成仓库 · 3层精准物理注入】
# =====================================================================

# 1. 精准向上翻 3 层，死死死锁定 'data-science-labs' 根目录
notebook_dir = os.getcwd() # 当前处于 .../03_datacamp/SQL/practice_lab
project_root = os.path.abspath(os.path.join(notebook_dir, "../../..")) # 🌟 正确翻 3 层

# 2. 直接锁定你现成的 raw_data 文件夹
raw_data_dir = os.path.join(project_root, "raw_data","datacamp","SQL","user_transactions")

# 3. 刚性指定数据库的绝对物理路径
db_path = os.path.join(raw_data_dir, "user_transaction.db")

print(f"📡 审查官绝对对账 - 正在物理连接你的现成仓库:\n👉 {db_path}")

# 4. 激活连接（这次绝对能进去了！）
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

cursor.execute("""
CREATE TABLE IF NOT EXISTS user_transactions (
    log_id INTEGER PRIMARY KEY AUTOINCREMENT,
    user_id TEXT,
    transaction_time TEXT,
    action_type TEXT,
    amount REAL
)
""")

# 5. 灌入数据
raw_data = [
    ('U02', '2026-06-18 09:00:00', 'click', 0.0),
    ('U02', '2026-06-18 09:15:00', 'purchase', 128.5),
    ('U02', '2026-06-18 10:30:00', 'purchase', 45.0),
    ('U03', '2026-06-18 09:05:00', None, 10.0),          
    ('U04', '2026-06-18 09:10:00', 'purchase', 20.0),      
    ('U01', '2026-06-18 09:01:00', 'purchase', 5.0),       
    ('U01', '2026-06-18 09:02:00', 'click', 0.0),          
    ('U01', '2026-06-18 09:05:00', 'purchase', 8.2),       
    ('U01', '2026-06-18 09:12:00', 'purchase', 12.0),      
    ('U01', '2026-06-18 09:28:00', 'purchase', 3.1),       
    ('U01', '2026-06-18 09:40:00', 'purchase', 50.0),      
]

df_seed = pd.DataFrame(raw_data, columns=['user_id', 'transaction_time', 'action_type', 'amount'])
df_seed.to_sql("user_transactions", conn, if_exists="replace", index=False, dtype={'log_id': 'INTEGER PRIMARY KEY AUTOINCREMENT'})
conn.commit()

print("\n🎉 警报解除！数据母体已完美、无损地注入到你现成的 raw_data 目录中！")

📡 审查官绝对对账 - 正在物理连接你的现成仓库:
👉 c:\projects\data-science-labs\raw_data\datacamp\SQL\user_transactions\user_transaction.db

🎉 警报解除！数据母体已完美、无损地注入到你现成的 raw_data 目录中！


### 🎯 最终要加工出来的 2 个特征字段：

最终落盘的 CSV 表里，除了保留原始的 `user_id` 和 `transaction_time` 外，必须多出下面这两个高价值特征列：

1. **`rolling_count_30m`**：
    
    - **含义**：**当前这笔交易前（包含当前这笔）的 30 分钟内，该用户一共发起了多少次 `'purchase'`（购买）行为？**
        
    - **反黑产原理**：如果这个数字很大（比如 30 分钟内买了 20 次），说明是自动化脚本在疯狂试探接口，属于高危黑产。
        
2. **`rolling_sum_amount_30m`**：
    
    - **含义**：**当前这笔交易前（包含当前这笔）的 30 分钟内，该用户累计刷卡消费了多少钱（`amount` 的总和）？**
        
    - **反黑产原理**：如果黑产在极短时间内把限额刷满了，或者高频小额累计到了某个危险阈值，系统需要立刻对该账户进行冻结。

In [2]:
# SQL轨道
sql_query = """
WITH last_30m_purchase_table AS (
SELECT  user_id,
        transaction_time,
        amount,
        COUNT(*)
        OVER(PARTITION BY user_id ORDER BY CAST(strftime('%s',datetime(transaction_time))AS INTEGER) ASC
        RANGE BETWEEN 1800 PRECEDING AND CURRENT ROW
        ) AS rolling_count_30m,
        SUM(amount)
        OVER(PARTITION BY user_id ORDER BY CAST(strftime('%s',datetime(transaction_time))AS INTEGER) ASC
        RANGE BETWEEN 1800 PRECEDING AND CURRENT ROW
        ) AS rolling_sum_amount_30m
FROM user_transactions 
where   action_type = 'purchase'
ORDER BY user_id ASC,transaction_time ASC
)
SELECT  user_id,
        transaction_time,
        rolling_count_30m,
        rolling_sum_amount_30m
FROM 
last_30m_purchase_table
"""
df_sql = pd.read_sql_query(sql_query,conn)
print(df_sql)

  user_id     transaction_time  rolling_count_30m  rolling_sum_amount_30m
0     U01  2026-06-18 09:01:00                  1                     5.0
1     U01  2026-06-18 09:05:00                  2                    13.2
2     U01  2026-06-18 09:12:00                  3                    25.2
3     U01  2026-06-18 09:28:00                  4                    28.3
4     U01  2026-06-18 09:40:00                  3                    65.1
5     U02  2026-06-18 09:15:00                  1                   128.5
6     U02  2026-06-18 10:30:00                  1                    45.0
7     U04  2026-06-18 09:10:00                  1                    20.0


In [3]:
#PANDAS轨道
df_raw = df_seed[df_seed['action_type'] == 'purchase'].copy()
df_raw['transaction_time'] = pd.to_datetime(df_raw['transaction_time'])

# 1. 物理激活时间宇宙：必须先全局排序，再将时间砸成 DatetimeIndex
df_time_index = df_raw.sort_values(by='transaction_time').set_index('transaction_time')

# 2. 独立战场计算滑窗，并立刻用 .reset_index() 物理降维
rolling_results = df_time_index.groupby('user_id')['amount'].rolling('30min').agg(['count','sum']).reset_index()

# 3. 🌟 组合双键无损焊接大盘
last_30m_purchase = pd.merge(
    df_raw,
    rolling_results,
    on = ['user_id', 'transaction_time'], 
    how = 'left'
).rename(columns={'count': 'rolling_count_30m', 'sum': 'rolling_sum_amount_30m'})

print(last_30m_purchase)

  user_id    transaction_time action_type  amount  rolling_count_30m  \
0     U02 2026-06-18 09:15:00    purchase   128.5                1.0   
1     U02 2026-06-18 10:30:00    purchase    45.0                1.0   
2     U04 2026-06-18 09:10:00    purchase    20.0                1.0   
3     U01 2026-06-18 09:01:00    purchase     5.0                1.0   
4     U01 2026-06-18 09:05:00    purchase     8.2                2.0   
5     U01 2026-06-18 09:12:00    purchase    12.0                3.0   
6     U01 2026-06-18 09:28:00    purchase     3.1                4.0   
7     U01 2026-06-18 09:40:00    purchase    50.0                3.0   

   rolling_sum_amount_30m  
0                   128.5  
1                    45.0  
2                    20.0  
3                     5.0  
4                    13.2  
5                    25.2  
6                    28.3  
7                    65.1  


In [9]:
# 流式ETL管道

# 1.建立刚性路径
import os
current_path = os.getcwd()

while os.path.basename(current_path) != 'data-science-labs':
    parent_path = os.path.dirname(current_path)
    if parent_path == current_path:
        break
    current_path = parent_path

folder_path = os.path.join(current_path,'raw_data','datacamp','SQL','user_transactions')
file_path = os.path.join(folder_path,'user_transactions.csv')

if not os.path.exists(folder_path):
    os.makedirs(folder_path)
if os.path.exists(file_path):
    os.remove(file_path)

# 2.SQL提取数据
sql_query = """
WITH last_30m_purchase_table AS (
SELECT  user_id,
        transaction_time,
        amount,
        COUNT(*)
        OVER(PARTITION BY user_id ORDER BY CAST(strftime('%s',datetime(transaction_time))AS INTEGER) ASC
        RANGE BETWEEN 1800 PRECEDING AND CURRENT ROW
        ) AS rolling_count_30m,
        SUM(amount)
        OVER(PARTITION BY user_id ORDER BY CAST(strftime('%s',datetime(transaction_time))AS INTEGER) ASC
        RANGE BETWEEN 1800 PRECEDING AND CURRENT ROW
        ) AS rolling_sum_amount_30m
FROM user_transactions 
where   action_type = 'purchase'
ORDER BY user_id ASC,transaction_time ASC
)
SELECT  user_id,
        transaction_time,
        rolling_count_30m,
        rolling_sum_amount_30m
FROM 
last_30m_purchase_table
"""
# 3.启用pandas流式管道
CHUNK_SIZE = 100000
total_records = 0

try:
    query_generator = pd.read_sql_query(sql_query,conn,chunksize=CHUNK_SIZE)

    for i,feature_chunk in enumerate(query_generator):
        if i == 0:
            feature_chunk.to_csv(file_path,mode='w',index=False)
        else:
            feature_chunk.to_csv(file_path,mode='a',header=False,index=False)
        
        current_rows = len(feature_chunk)
        total_records += current_rows
        print(
            f"处理了{i+1}批次"
            f"本批落盘{current_rows}条，系统累计放行总数据{total_records}条"
        )
except Exception as e:
    print(f"【重大安全事故】投产管道运行崩溃{e}")

try:
    df_result = pd.read_csv(file_path)
    print(df_result.to_string(index=False))
except FileNotFoundError:
    print(f"当前工作目录未找到文件{file_path},请检查路径")

处理了1批次本批落盘8条，系统累计放行总数据8条
user_id    transaction_time  rolling_count_30m  rolling_sum_amount_30m
    U01 2026-06-18 09:01:00                  1                     5.0
    U01 2026-06-18 09:05:00                  2                    13.2
    U01 2026-06-18 09:12:00                  3                    25.2
    U01 2026-06-18 09:28:00                  4                    28.3
    U01 2026-06-18 09:40:00                  3                    65.1
    U02 2026-06-18 09:15:00                  1                   128.5
    U02 2026-06-18 10:30:00                  1                    45.0
    U04 2026-06-18 09:10:00                  1                    20.0
